<a href="https://colab.research.google.com/github/data4class/Teaching/blob/main/Deep_Learning_Demo_1_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
from sklearn.cluster import SpectralClustering
from sklearn.preprocessing import StandardScaler

def get_nifty500_symbols_and_industries(): # Ex: Make this function modular to get data for any Index.
    url = "https://archives.nseindia.com/content/indices/ind_nifty500list.csv"
    df = pd.read_csv(url)
    symbols = [f"{symbol}.NS" for symbol in df['Symbol'].tolist()]
    industries = df['Industry'].tolist()
    return symbols, industries

def download_data(symbols, years = 5):
    end_date = datetime.now()
    start_date = end_date - timedelta(days=years*365)  # 5 years ago
    all_data = {}
    for symbol in symbols:
        try:
            stock = yf.Ticker(symbol)
            data = stock.history(start=start_date, end=end_date)
            # Ex: Remove securities other than Equity (EQ series)
            if not data.empty:
                all_data[symbol] = data['Close']
            else:
                print(f"No data available for {symbol}")
#            time.sleep(1)  # To avoid hitting rate limits
        except Exception as e:
            print(f"Error downloading data for {symbol}: {e}")

    return pd.DataFrame(all_data)

# Get the list of Nifty 500 symbols and industries
nifty500_symbols, nifty500_industries = get_nifty500_symbols_and_industries()

# Download the data
print("Downloading data...")
df = download_data(nifty500_symbols, 2)

# Drop stocks with more than 5 days of missing data
max_allowed_missing = 5
columns_to_keep = df.columns[df.isna().sum() <= max_allowed_missing]
df_filtered = df[columns_to_keep]

# Fill remaining missing data using bfill
df_filled = df_filtered.bfill() # Ex: Use cubic spline to fill missing values.

# Compute daily returns
returns = df_filled.pct_change().dropna()



In [ ]:
print(nifty500_symbols)
print(nifty500_industries)
print(df_filled.head())
print(returns.head())

In [ ]:
# Compute the covariance matrix
# Ex: Try computing distance between two column vectors (distance between two stock).
# Here we are treating daily returns as cordinates in d (number of days) dimension.
corr_matrix = returns.corr()
import math
def f(x):
    return math.e**(-(math.acos(x)**2)/2)
#Ex: weight_matrix = (corr_matrix + 1) / 2
# Create the weight matrix
weight_matrix = corr_matrix.applymap(f)

In [ ]:
print(weight_matrix.head())

In [ ]:
# Get the number of unique industries
# Try either elbow or silhoute method to compute ideal number of clusters.
#n_clusters = len(set(nifty500_industries))

n_clusters = 20

print(f"Number of clusters (unique industries): {n_clusters}")
print(f"Number of stocks after cleaning: {len(returns.columns)}")

# Apply Spectral Clustering
spectral = SpectralClustering(n_clusters=n_clusters, affinity='precomputed', random_state=42)
cluster_labels = spectral.fit_predict(weight_matrix)

# Create a DataFrame with clustering results
clustering_results = pd.DataFrame({
    'Symbol': returns.columns,
    'Cluster': cluster_labels
})

# Print cluster information
for i in range(n_clusters):
    cluster_stocks = clustering_results[clustering_results['Cluster'] == i]['Symbol'].tolist()
    print(f"\nCluster {i} stocks ({len(cluster_stocks)}):")
    print(", ".join(cluster_stocks[:5]) + ("..." if len(cluster_stocks) > 5 else ""))

# Optional: Evaluate cluster quality
# Use this to also evaluate optimal number of clusters
from sklearn.metrics import silhouette_score
silhouette_avg = silhouette_score(weight_matrix, cluster_labels)
print(f"\nSilhouette Score: {silhouette_avg}")

# Save clustering results
clustering_results.to_csv("nifty500_clustering_results.csv", index=False)
print("\nClustering results saved to nifty500_clustering_results.csv")

In [ ]:
# Calculate variance for each stock
variances = returns.var()

# Function to select stock with least variance from a cluster
def select_least_variance_stock(cluster_stocks):
    cluster_variances = variances[cluster_stocks]
    return cluster_variances.idxmin()

# Select one stock from each cluster with the least variance
selected_portfolio = {}
for cluster in range(n_clusters):
    cluster_stocks = clustering_results[clustering_results['Cluster'] == cluster]['Symbol'].tolist()
    if cluster_stocks:
        selected_stock = select_least_variance_stock(cluster_stocks)
        selected_portfolio[cluster] = selected_stock

# Print selected portfolio
print("\nSelected Portfolio:")
for cluster, stock in selected_portfolio.items():
    print(f"Cluster {cluster}: {stock} (Variance: {variances[stock]:.6f})")

# Calculate portfolio statistics
portfolio_returns = returns[list(selected_portfolio.values())]
portfolio_mean_return = portfolio_returns.mean()
portfolio_volatility = portfolio_returns.std()

print("\nPortfolio Statistics:")
print(f"Number of stocks in portfolio: {len(selected_portfolio)}")
print(f"Average Daily Return: {portfolio_mean_return.mean():.6f}")
print(f"Portfolio Volatility: {portfolio_volatility.mean():.6f}")

# Optional: Calculate and print correlation matrix of selected stocks
correlation_matrix = portfolio_returns.corr()
print("\nCorrelation Matrix of Selected Stocks:")
print(correlation_matrix)

# Save selected portfolio to CSV
pd.DataFrame(list(selected_portfolio.items()), columns=['Cluster', 'Selected Stock']).to_csv("selected_portfolio.csv", index=False)
print("\nSelected portfolio saved to selected_portfolio.csv")

# Evaluate overall cluster quality
silhouette_avg = silhouette_score(weight_matrix, cluster_labels)
print(f"\nOverall Silhouette Score: {silhouette_avg}")

Exercise: Compute sharpe ratio for these three portfolio with equal weights.
1. Portfolio using industry clusters.
2. Using (1+corr)/2 method.
3. Using e^(-(acos(corr)^2)/2) method.